In [0]:
#Reads the credentials from the Databricks Secret Scope
import os
from google.cloud import bigquery
from google.oauth2 import service_account

gcp_key_json = dbutils.secrets.get(scope="gcp-bigquery", key="service-account-key")

with open("/tmp/gcp_key.json", "w") as f:
    f.write(gcp_key_json)

#Stores the credentials for later use
credentials = service_account.Credentials.from_service_account_file("/tmp/gcp_key.json")
client = bigquery.Client(credentials=credentials, project=credentials.project_id)

In [0]:
import time

#Path configuration
LANDING_PATH = "/Volumes/dbw_techchallenge/staging/landing_alunos"
CHECKPOINT_PATH = "/Volumes/dbw_techchallenge/bronze/checkpoints"
spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_techchallenge.staging")
spark.sql("CREATE VOLUME IF NOT EXISTS dbw_techchallenge.staging.landing_alunos")
spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_techchallenge.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS dbw_techchallenge.bronze.checkpoints")

#Simulates an incoming event
student_pool_df = spark.table("dbw_techchallenge.staging.alunos_pool").toPandas()
batch_df = student_pool_df.sample(n=50)
event_timestamp = int(time.time())
event_file_path = f"{LANDING_PATH}/event_{event_timestamp}.json"

#Prints the simulated event
batch_df.to_json(event_file_path, orient="records", lines=True)
print(f"Batch of {len(batch_df)} records written to {event_file_path}")

#Ingests streamed data
stream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/students_schema")
    .load(LANDING_PATH))

stream_query = (stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/students_bronze")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("dbw_techchallenge.bronze.alunos_stream"))

stream_query.awaitTermination()

#Confirmation
total_rows = spark.sql(
    "SELECT count(*) as total FROM dbw_techchallenge.bronze.alunos_stream"
).collect()[0]["total"]
print(f"Ingestion complete. Total accumulated in bronze: {total_rows} rows")